# Финальные модели Сравнение не только с ласт кликом


In [ ]:
import math
import pandas as pd
import numpy as np
import itertools
import random
from collections import defaultdict

def compute_conversion_prob_smooth(journey_paths, alpha=1.0, max_iter=1000, tol=1e-6):
    """
    Вычисляет вероятность конверсии через итерационный метод Марковских цепей с Laplace smoothing.
    """
    transitions = {}
    for path in journey_paths:
        prev = 'start'
        for state in path:
            transitions[(prev, state)] = transitions.get((prev, state), 0) + 1
            prev = state

    states = set()
    transitions_possible = {}
    for (s, t) in transitions.keys():
        states.add(s)
        states.add(t)
        if s not in transitions_possible:
            transitions_possible[s] = set()
        transitions_possible[s].add(t)

    p = {state: 0.0 for state in states}
    p['conversion'] = 1.0
    p['null'] = 0.0

    for _ in range(max_iter):
        p_old = p.copy()
        diff = 0
        for s in states:
            if s in ['conversion', 'null']:
                continue
            out_states = list(transitions_possible.get(s, []))
            if not out_states:
                continue
            total = sum(transitions.get((s, t), 0) for t in out_states)
            k = len(out_states)
            p[s] = sum(((transitions.get((s, t), 0) + alpha) / (total + alpha * k)) * p_old[t] for t in out_states)
            diff = max(diff, abs(p[s] - p_old[s]))
        if diff < tol:
            break

    return p.get('start', 0.0)


f_cache = {}
def f_of_S(S, journeys_data, global_rate):
    """
    Эмпирическая оценка конверсионной вероятности для путей, где S ⊆ (множество каналов).
    """
    if S in f_cache:
        return f_cache[S]
    eligible = [conv for (journey_set, conv) in journeys_data if S.issubset(journey_set)]
    result = np.mean(eligible) if eligible else global_rate
    f_cache[S] = result
    return result

def compute_shapley_attribution(journeys_data, global_conversion_rate, num_permutations=1000):
    """
    Вычисляет атрибуцию методом Шэпли для каждого канала.
    """
    shapley_credit = defaultdict(float)
    for journey_set, conv in journeys_data:
        if conv == 0:
            continue
        channels_list = list(journey_set)
        n = len(channels_list)
        if n == 0:
            continue
        if math.factorial(n) <= num_permutations:
            perms = list(itertools.permutations(channels_list))
        else:
            perms = [tuple(random.sample(channels_list, n)) for _ in range(num_permutations)]
        journey_contrib = defaultdict(float)
        for perm in perms:
            current_set = frozenset()
            for ch in perm:
                S_without = current_set
                S_with = frozenset(list(current_set) + [ch])
                marginal = f_of_S(S_with, journeys_data, global_conversion_rate) - f_of_S(S_without, journeys_data, global_conversion_rate)
                journey_contrib[ch] += marginal
                current_set = S_with
        for ch in journey_contrib:
            shapley_credit[ch] += journey_contrib[ch] / len(perms)
    return shapley_credit


visits = pd.read_csv('visits.csv')
orders = pd.read_csv('orders.csv')


visits['Session Start'] = pd.to_datetime(visits['Session Start'])
orders['Event Dt'] = pd.to_datetime(orders['Event Dt'])


user_duration = visits.groupby('User Id')['Session Start'].agg(['min', 'max'])
user_duration['duration'] = user_duration['max'] - user_duration['min']
valid_users = user_duration[user_duration['duration'] <= pd.Timedelta(days=30)].index
visits = visits[visits['User Id'].isin(valid_users)]
orders = orders[orders['User Id'].isin(valid_users)]

orders_sorted = orders.sort_values(by=['User Id', 'Event Dt'])
first_purchases = orders_sorted.drop_duplicates(subset=['User Id'], keep='first').reset_index(drop=True)


merged_first_purchases = pd.merge(visits, first_purchases[['User Id', 'Event Dt']], on='User Id', how='left')
merged_first_purchases['is_paid'] = merged_first_purchases['Event Dt'] >= merged_first_purchases['Session Start']
merged_first_purchases['is_false'] = merged_first_purchases['Event Dt'].isna()
merged_first_purchases['is_valid'] = (merged_first_purchases['is_paid'] != False) | (merged_first_purchases['is_false'] != False)
visits = merged_first_purchases[merged_first_purchases['is_valid'] == True].copy()


converted_users = set(first_purchases['User Id'].unique())

journey_paths = []
journeys_set = []
conversion_label = []
last_click_counter = defaultdict(int)
first_click_counter = defaultdict(int)

for user, group in visits.groupby('User Id'):
    group_sorted = group.sort_values('Session Start')
    path = group_sorted['Channel'].tolist()
    channels_set = frozenset(path)
    if user in converted_users:
        journey_paths.append(path + ['conversion'])
        journeys_set.append(channels_set)
        conversion_label.append(1)
        if path:
            last_click_counter[path[-1]] += 1
            first_click_counter[path[0]] += 1
    else:
        journey_paths.append(path + ['null'])
        journeys_set.append(channels_set)
        conversion_label.append(0)

total_conversions = sum(conversion_label)
journeys_data = list(zip(journeys_set, conversion_label))
global_conversion_rate = np.mean(conversion_label)


baseline_conversion = compute_conversion_prob_smooth(journey_paths, alpha=1.0)
print("Базовая конверсионная вероятность (Markov, smoothed):", baseline_conversion)

channels = visits['Channel'].unique().tolist()
removal_effect = {}
for channel in channels:
    modified_paths = []
    for path in journey_paths:
        modified = [state for state in path if state != channel]
        if not modified:
            modified = ['null']
        modified_paths.append(modified)
    conv_prob_removed = compute_conversion_prob_smooth(modified_paths, alpha=1.0)
    removal_effect[channel] = baseline_conversion - conv_prob_removed

min_effect = min(removal_effect.values())
if min_effect < 0:
    removal_effect = {channel: v - min_effect for channel, v in removal_effect.items()}

total_removal = sum(removal_effect.values())
markov_attr = {channel: (removal_effect[channel] / total_removal if total_removal != 0 else 0) for channel in channels}
markov_predicted_conversions = {ch: markov_attr[ch] * total_conversions for ch in channels}

print("\nМарковская атрибуция по каналам (улучшенная + коррекция):")
for ch, val in markov_predicted_conversions.items():
    print(f"{ch}: {val:.2f} конверсий")

# Шэпли атрибуция
f_cache.clear()
shapley_credit = compute_shapley_attribution(journeys_data, global_conversion_rate, num_permutations=1000)
total_shapley = sum(shapley_credit.values())
print("\nСуммарный вклад по Шэпли (ожидаемые конверсии, улучшенная):", total_shapley)
shapley_attr = {ch: (shapley_credit.get(ch, 0) / total_shapley if total_shapley != 0 else 0) for ch in channels}
shapley_predicted_conversions = {ch: shapley_attr[ch] * total_conversions for ch in channels}

print("\nАтрибуция методом Шэпли по каналам (улучшенная):")
for ch, val in shapley_predicted_conversions.items():
    print(f"{ch}: {val:.2f} конверсий")

# Базовые атрибуции:
print("\nLast-click атрибуция (конверсии по последнему клику):")
for ch in channels:
    print(f"{ch}: {last_click_counter.get(ch, 0)} конверсий")

print("\nFirst-click атрибуция (конверсии по первому касанию):")
for ch in channels:
    print(f"{ch}: {first_click_counter.get(ch, 0)} конверсий")



def compute_regression_metrics(observed, predicted):
    keys = list(observed.keys())
    obs = np.array([observed[k] for k in keys])
    pred = np.array([predicted.get(k, 0) for k in keys])
    mse = np.mean((obs - pred) ** 2)
    ss_res = np.sum((obs - pred) ** 2)
    ss_tot = np.sum((obs - np.mean(obs)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan
    return mse, r2

print("\nМетрики качества (сравнение с базовыми атрибуциями):")

print("\nСравнение с Last-click атрибуцией:")
observed_last_click = dict(last_click_counter)
mse_markov, r2_markov = compute_regression_metrics(observed_last_click, markov_predicted_conversions)
mse_shapley, r2_shapley = compute_regression_metrics(observed_last_click, shapley_predicted_conversions)
print("Марковская модель: MSE = {:.4f}, R² = {:.4f}".format(mse_markov, r2_markov))
print("Шэпли модель:     MSE = {:.4f}, R² = {:.4f}".format(mse_shapley, r2_shapley))

print("\nСравнение с First-click атрибуцией:")
observed_first_click = dict(first_click_counter)
mse_markov_fc, r2_markov_fc = compute_regression_metrics(observed_first_click, markov_predicted_conversions)
mse_shapley_fc, r2_shapley_fc = compute_regression_metrics(observed_first_click, shapley_predicted_conversions)
print("Марковская модель: MSE = {:.4f}, R² = {:.4f}".format(mse_markov_fc, r2_markov_fc))
print("Шэпли модель:     MSE = {:.4f}, R² = {:.4f}".format(mse_shapley_fc, r2_shapley_fc))


Базовая конверсионная вероятность (Markov, smoothed): 0.033054422322483915

Марковская атрибуция по каналам (улучшенная + коррекция):
organic: 484.03 конверсий
TipTop: 433.53 конверсий
RocketSuperAds: 491.23 конверсий
YRabbit: 506.73 конверсий
FaceBoom: 0.00 конверсий
MediaTornado: 504.96 конверсий
AdNonSense: 242.45 конверсий
LeapBob: 531.78 конверсий
WahooNetBanner: 510.75 конверсий
OppleCreativeMedia: 533.02 конверсий
lambdaMediaAds: 490.54 конверсий

Суммарный вклад по Шэпли (ожидаемые конверсии, улучшенная): 275.09944850385

Атрибуция методом Шэпли по каналам (улучшенная):
organic: -115.57 конверсий
TipTop: -75.60 конверсий
RocketSuperAds: -22.59 конверсий
YRabbit: -19.18 конверсий
FaceBoom: 4725.61 конверсий
MediaTornado: -21.63 конверсий
AdNonSense: 353.94 конверсий
LeapBob: -27.75 конверсий
WahooNetBanner: -34.93 конверсий
OppleCreativeMedia: -24.84 конверсий
lambdaMediaAds: -8.45 конверсий

Last-click атрибуция (конверсии по последнему клику):
organic: 221 конверсий
TipTop: 41

In [ ]:
print("\nФинальные веса атрибуции по каналам (Markov model):")
for channel, weight in markov_attr.items():
    print(f"{channel}: {weight:.2%}")

print("\nФинальные веса атрибуции по каналам (Shapley model):")
for channel, weight in shapley_attr.items():
    print(f"{channel}: {weight:.2%}")


Финальные веса атрибуции по каналам (Markov model):
organic: 10.24%
TipTop: 9.17%
RocketSuperAds: 10.39%
YRabbit: 10.72%
FaceBoom: 0.00%
MediaTornado: 10.68%
AdNonSense: 5.13%
LeapBob: 11.25%
WahooNetBanner: 10.80%
OppleCreativeMedia: 11.27%
lambdaMediaAds: 10.37%

Финальные веса атрибуции по каналам (Shapley model):
organic: -2.44%
TipTop: -1.60%
RocketSuperAds: -0.48%
YRabbit: -0.41%
FaceBoom: 99.93%
MediaTornado: -0.46%
AdNonSense: 7.48%
LeapBob: -0.59%
WahooNetBanner: -0.74%
OppleCreativeMedia: -0.53%
lambdaMediaAds: -0.18%


In [ ]:
#### Если перенормировать и убрать отрицательные значения получим FaceBoom: примерно 93.1%

####AdNonSense: примерно 7.0% по модели шэпли